<a href="https://colab.research.google.com/github/mhmddirany/Video-Transcription-Hebrew-ASR-Pipeline/blob/main/notebooks/models_evaluation_data1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q git+https://github.com/QwenLM/Qwen3-ASR.git
!pip install -q jiwer librosa soundfile
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.8/416.8 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 153.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 156.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.2 MB/s eta 0:00:00
G

In [ ]:
import os, json, pandas as pd, librosa, numpy as np

BASE      = "/content/drive/MyDrive/saspeech_project/extracted/saspeech_gold_standard"
WAV_DIR   = os.path.join(BASE, "wavs")
meta_path = os.path.join(BASE, "metadata.csv")

df = pd.read_csv(meta_path, sep="|", header=None, engine="python")
df.columns = ["file_id", "text", "extra"] if len(df.columns) == 3 else ["file_id", "text"]
print("Total rows:", len(df))
print(df.head())

# Build dataset list (same 100 samples as before)
dataset = []
for _, row in df.iterrows():
    wav_path = os.path.join(WAV_DIR, str(row["file_id"]) + ".wav")
    if os.path.exists(wav_path):
        dataset.append({"file": str(row["file_id"]), "audio_path": wav_path, "reference": str(row["text"])})

print(f"Found {len(dataset)} wav files")
dataset = dataset[:100]   # same 100 as before
print(f"Using first 100 samples")

# Print total duration of 100 samples
total_dur = sum(librosa.get_duration(path=s["audio_path"]) for s in dataset)
print(f"Total duration: {total_dur/60:.1f} min")

Total rows: 2986
             file_id                                               text  \
0  gold_000_line_000                         שָׁלוֹם, צְלִיל אַבְרָהָם.   
1  gold_000_line_001                          לְגַמְרֵי, מַדְהִים, לֹא?   
2  gold_000_line_002  וְדַוְוקָא בִּגְלַל שֶׁכּוּלָּנוּ הָיִינוּ עֲס...   
3  gold_000_line_003  אָז הַיּוֹם אֲנַחְנוּ נְדַבֵּר עַל הַחֲקִירָה ...   
4  gold_000_line_004  הָרָמַת מָסַךְ מֵעַל סְבַךְ שֶׁל אִינְטֶרֶסִים...   

                                               extra  
0                         שָׁלוֹם, צְלִיל אַבְרָהָם.  
1                          לְגַמְרֵי, מַדְהִים, לֹא?  
2  וְדַוְוקָא בִּגְלַל שֶׁכּוּלָּנוּ הָיִינוּ עֲס...  
3  אָז הַיּוֹם אֲנַחְנוּ נְדַבֵּר עַל הַחֲקִירָה ...  
4  הָרָמַת מָסַךְ מֵעַל סְבַךְ שֶׁל אִינְטֶרֶסִים...  
Found 2986 wav files
Using first 100 samples
Total duration: 8.1 min


In [ ]:
import gc, time, torch, numpy as np, librosa
from qwen_asr import Qwen3ASRModel
from jiwer import wer, cer
import re, unicodedata

device_map = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype      = torch.float16 if torch.cuda.is_available() else torch.float32

print("Loading Caspi (OzLabs/Caspi-1.7B) ...")
caspi_model = Qwen3ASRModel.from_pretrained(
    "OzLabs/Caspi-1.7B",
    dtype=dtype,
    device_map=device_map,
    max_inference_batch_size=1,
    max_new_tokens=256,
)
print("✅ Caspi loaded\n")

def normalize_hebrew(text):
    if not isinstance(text, str): return ''
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'[\u0591-\u05C7]', '', text)
    text = re.sub(r'[^\w\sא-ת]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

results_caspi = []
t_start = time.perf_counter()

for i, item in enumerate(dataset):
    audio, sr = librosa.load(item["audio_path"], sr=16000, mono=True)
    t0     = time.perf_counter()
    result = caspi_model.transcribe(audio=(audio, sr), language=None)
    rtf    = (time.perf_counter() - t0) / (len(audio)/sr)
    hyp    = result[0].text.strip()

    results_caspi.append({
        "file":      item["file"],
        "reference": item["reference"],
        "hypothesis": hyp,
        "rtf":       round(rtf, 3),
    })

    print(f"[{i+1}/100] RTF={rtf:.2f}x")
    print(f"  REF: {item['reference'][:80]}")
    print(f"  HYP: {hyp[:80]}")

    del audio, result
    gc.collect()
    torch.cuda.empty_cache()

wall = time.perf_counter() - t_start
print(f"\n✅ Done — {wall/60:.1f} min")

Loading Caspi (OzLabs/Caspi-1.7B) ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.08G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

...

[97/100] RTF=0.19x
  REF: בִּכְלָל יָדַעְנוּ עַל הַקֶּשֶׁר הַזֶּה בֵּין אַלוֹבִיץ', אָה, אָה, וּבַנְק הַפּ
  HYP: בכלל ידענו על הקשר הזה בין אלוביץ' ובנק הפועלים.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[98/100] RTF=0.27x
  REF: נָכוֹן. וְז-, וְזֶה חֵלֶק דֵּי מַדְהִים, הַסִּיפּוּר הַזֶּה. שׁוּב, אֲנַחְנוּ לֹ
  HYP: נכון, וזה חלק די מדהים מהסיפור הזה, שוב, אנחנו לא משפטנים, אז אין לנו איזושהי יכ
[99/100] RTF=0.35x
  REF: אָה, עַד כַּמָּה זֶה פּוֹגֵעַ בְּקַיְיס שֶׁל הַתְּבִיעָה, כֵּן אוֹ לֹא. אֲבָל,
  HYP: עד כמה זה פוגע בקייט של התביעה, כן או לא, אבל.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[100/100] RTF=0.27x
  REF: אֲנִי חַיָּיב לְהַגִּיד שֶׁכְּצוֹפֶה מִן הַצַּד וּכְמִי שֶׁמִּתְעַנְיֵין בַּיְּח
  HYP: אני חייב להגיד שכצופה מן הצד וכמי שמתעניין ביחסים האלה שבין העולם העסקי ועולם הת

✅ Done — 4.1 min


In [ ]:
from jiwer import wer, cer

refs = [normalize_hebrew(r["reference"]) for r in results_caspi]
hyps = [normalize_hebrew(r["hypothesis"]) for r in results_caspi]

caspi_wer = round(wer(refs, hyps) * 100, 2)
caspi_cer = round(cer(refs, hyps) * 100, 2)

print("\n" + "="*55)
print(f"  {'Model':<30} {'WER %':>7} {'CER %':>7}")
print("─"*55)
print(f"  {'Whisper large-v3':<30} {'15.12':>7} {'7.06':>7}")
print(f"  {'SeamlessM4T v2 large':<30} {'17.83':>7} {'8.67':>7}")
print(f"  {'ivrit-ai/whisper-large-v3':<30} {'11.24':>7} {'5.73':>7}")
print(f"  {'Caspi (OzLabs/Caspi-1.7B)':<30} {caspi_wer:>7} {caspi_cer:>7}")
print("="*55)
print(f"\n(100 samples | {total_dur/60:.1f} min of audio)")

# Save results
import json, os
SAVE_DIR = "/content/drive/MyDrive/saspeech_project/results"
os.makedirs(SAVE_DIR, exist_ok=True)
with open(os.path.join(SAVE_DIR, "caspi_results_100.json"), "w", encoding="utf-8") as f:
    json.dump(results_caspi, f, ensure_ascii=False, indent=2)
print("Results saved to Drive.")


  Model                            WER %   CER %
───────────────────────────────────────────────────────
  Whisper large-v3                 15.12    7.06
  SeamlessM4T v2 large             17.83    8.67
  ivrit-ai/whisper-large-v3        11.24    5.73
  Caspi (OzLabs/Caspi-1.7B)         14.5     6.9

(100 samples | 8.1 min of audio)
Results saved to Drive.


In [4]:
import json
from pathlib import Path

path = Path("/content/drive/MyDrive/saspeech_project/models_evaluation_data1.ipynb")  # change this to the printed path

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

nb.get("metadata", {}).pop("widgets", None)

for cell in nb.get("cells", []):
    cell.get("metadata", {}).pop("widgets", None)
    for output in cell.get("outputs", []):
        output.get("metadata", {}).pop("widgets", None)

with open(path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("Notebook cleaned:", path)

Notebook cleaned: /content/drive/MyDrive/saspeech_project/models_evaluation_data1.ipynb
